# Task 2: SQL & Data Extraction
**ApexPlanet Data Analytics Internship**

Objective: master SQL for extraction, transformation, and aggregation, and connect Python to a database.

Dataset: cleaned Amazon Sale Report (128,942 order lines, Mar-Jun 2022), the same file produced in Task 1.

Structure of this notebook mirrors the task brief:
1. SQL fundamentals (SELECT/WHERE/ORDER BY/LIMIT, JOINs, GROUP BY/HAVING, subqueries/CTEs, window functions)
2. Advanced SQL for business questions (trends, top locations, retention, category performance, moving averages, views, query optimization)
3. Python + SQL integration (SQLAlchemy, `pandas.read_sql`, parameterized queries, the reusable `db_utils.py` module)

> **Note on "customers":** this Amazon seller report has no `customer_id` column (Amazon does not expose one). Wherever the task brief asks for "customer" analysis, `ship_city` + `ship_postal_code` is used as the closest available proxy for a delivery location, and this is called out again at the point it's used.

## 0. Setup
Run `scripts/load_data.py` once beforehand to build `data/db/amazon_sales.db` from the cleaned CSV.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../scripts"))

import pandas as pd
from db_utils import get_engine, run_query, execute, explain, table_exists

pd.set_option("display.max_columns", None)

engine = get_engine()
print("Connected to:", engine.url)
assert table_exists("sales"), "Run scripts/load_data.py first to build the database."


## 1. SQL Fundamentals (Day 9-11)

### 1.1 SELECT / WHERE / ORDER BY / LIMIT
Ten highest-value shipped orders in the *Western Dress* category.

In [ ]:
sql = """
SELECT order_id, date, category, qty, amount, ship_state
FROM sales
WHERE category = :category
  AND status LIKE 'Shipped%'
ORDER BY amount DESC
LIMIT :n
"""
run_query(sql, {"category": "Western Dress", "n": 10})


### 1.2-1.5 JOIN types
A small `state_region` lookup table (built in `scripts/load_data.py`) maps each shipping state to a region, so we can demonstrate INNER / LEFT / RIGHT / FULL joins against the `sales` table.

In [ ]:
inner_join_sql = """
SELECT s.order_id, s.ship_state, r.region, s.amount
FROM sales AS s
INNER JOIN state_region AS r
    ON UPPER(TRIM(s.ship_state)) = r.state_norm
LIMIT 10
"""
run_query(inner_join_sql)


In [ ]:
left_join_sql = """
SELECT s.order_id, s.ship_state, r.region, s.amount
FROM sales AS s
LEFT JOIN state_region AS r
    ON UPPER(TRIM(s.ship_state)) = r.state_norm
WHERE r.region IS NULL
LIMIT 10
"""
# Any rows returned here point to a ship_state spelling the lookup table doesn't cover yet.
run_query(left_join_sql)


In [ ]:
right_join_sql = """
SELECT r.region, r.state_norm, s.order_id
FROM sales AS s
RIGHT JOIN state_region AS r
    ON UPPER(TRIM(s.ship_state)) = r.state_norm
LIMIT 10
"""
run_query(right_join_sql)


In [ ]:
full_join_sql = """
SELECT s.order_id, s.ship_state, r.region
FROM sales AS s
FULL OUTER JOIN state_region AS r
    ON UPPER(TRIM(s.ship_state)) = r.state_norm
WHERE s.order_id IS NULL OR r.region IS NULL
LIMIT 10
"""
# An empty result here is a good sign: every ship_state in the data maps to a region.
run_query(full_join_sql)


### 1.6-1.7 GROUP BY, aggregates, HAVING

In [ ]:
category_summary_sql = """
SELECT
    category,
    COUNT(*) AS order_count,
    SUM(qty) AS total_units,
    ROUND(AVG(amount), 2) AS avg_order_value,
    MIN(amount) AS min_order_value,
    MAX(amount) AS max_order_value
FROM sales
WHERE amount > 0
GROUP BY category
ORDER BY order_count DESC
"""
run_query(category_summary_sql)


In [ ]:
high_value_states_sql = """
SELECT
    ship_state,
    COUNT(*) AS order_count,
    ROUND(AVG(amount), 2) AS avg_order_value
FROM sales
WHERE amount > 0
GROUP BY ship_state
HAVING COUNT(*) > 500 AND AVG(amount) > 600
ORDER BY avg_order_value DESC
"""
run_query(high_value_states_sql)


### 1.8-1.9 Subqueries and CTEs

In [ ]:
above_avg_sql = """
SELECT order_id, category, amount
FROM sales
WHERE amount > (SELECT AVG(amount) FROM sales WHERE amount > 0)
ORDER BY amount DESC
LIMIT 10
"""
run_query(above_avg_sql)


In [ ]:
above_avg_by_category_sql = """
WITH overall_avg AS (
    SELECT AVG(amount) AS avg_amount FROM sales WHERE amount > 0
)
SELECT category, COUNT(*) AS above_avg_orders
FROM sales
WHERE amount > (SELECT avg_amount FROM overall_avg)
GROUP BY category
ORDER BY above_avg_orders DESC
"""
run_query(above_avg_by_category_sql)


### 1.10 Window functions: ROW_NUMBER, RANK, LAG, LEAD

In [ ]:
window_sql = """
SELECT
    category,
    order_id,
    amount,
    ROW_NUMBER() OVER (PARTITION BY category ORDER BY amount DESC) AS row_num,
    RANK()       OVER (PARTITION BY category ORDER BY amount DESC) AS rank_num,
    LAG(amount)  OVER (PARTITION BY category ORDER BY amount DESC) AS prev_amount,
    LEAD(amount) OVER (PARTITION BY category ORDER BY amount DESC) AS next_amount
FROM sales
WHERE amount > 0
ORDER BY category, row_num
LIMIT 15
"""
run_query(window_sql)


## 2. Advanced SQL for Business Questions (Day 12-14)

### 2.1 Monthly sales trend

In [ ]:
monthly_trend_sql = """
SELECT
    order_month,
    COUNT(*) AS order_count,
    SUM(amount) AS total_revenue,
    ROUND(AVG(amount), 2) AS avg_order_value
FROM sales
WHERE amount > 0
GROUP BY order_month
ORDER BY order_month
"""
monthly_trend = run_query(monthly_trend_sql)
monthly_trend


In [ ]:
monthly_trend.plot(x="order_month", y="total_revenue", kind="bar", legend=False,
                    title="Total Revenue by Month", figsize=(8, 4))


### 2.2 Top 10 "customers" by revenue
Reminder: no `customer_id` exists in this export, so `ship_city` + `ship_postal_code` stands in for a delivery location/customer proxy.

In [ ]:
top_customers_sql = """
SELECT
    ship_city,
    ship_postal_code,
    COUNT(*) AS order_count,
    SUM(amount) AS total_revenue
FROM sales
WHERE amount > 0
GROUP BY ship_city, ship_postal_code
ORDER BY total_revenue DESC
LIMIT 10
"""
run_query(top_customers_sql)


### 2.3 "Customer" retention rate (same proxy caveat as above)
Share of ship-to locations with orders in more than one distinct `order_week`.

In [ ]:
retention_sql = """
WITH location_weeks AS (
    SELECT ship_city, ship_postal_code, COUNT(DISTINCT order_week) AS active_weeks
    FROM sales
    WHERE amount > 0
    GROUP BY ship_city, ship_postal_code
)
SELECT
    COUNT(*) AS total_locations,
    SUM(CASE WHEN active_weeks > 1 THEN 1 ELSE 0 END) AS returning_locations,
    ROUND(100.0 * SUM(CASE WHEN active_weeks > 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS retention_rate_pct
FROM location_weeks
"""
run_query(retention_sql)


### 2.4 Product category performance

In [ ]:
category_perf_sql = """
SELECT
    category,
    COUNT(*) AS order_count,
    SUM(qty) AS units_sold,
    SUM(amount) AS total_revenue,
    ROUND(AVG(amount), 2) AS avg_order_value,
    ROUND(100.0 * SUM(amount) / (SELECT SUM(amount) FROM sales WHERE amount > 0), 2) AS pct_of_total_revenue
FROM sales
WHERE amount > 0
GROUP BY category
ORDER BY total_revenue DESC
"""
run_query(category_perf_sql)


### 2.5 Moving average and cumulative sum of daily revenue

In [ ]:
moving_avg_sql = """
WITH daily AS (
    SELECT order_day, SUM(amount) AS daily_revenue
    FROM sales
    WHERE amount > 0
    GROUP BY order_day
)
SELECT
    order_day,
    daily_revenue,
    ROUND(AVG(daily_revenue) OVER (
        ORDER BY order_day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) AS revenue_7day_moving_avg,
    SUM(daily_revenue) OVER (ORDER BY order_day ROWS UNBOUNDED PRECEDING) AS revenue_cumulative
FROM daily
ORDER BY order_day
"""
daily_trend = run_query(moving_avg_sql)
daily_trend.tail(10)


In [ ]:
ax = daily_trend.set_index("order_day")[["daily_revenue", "revenue_7day_moving_avg"]].plot(
    figsize=(11, 4), title="Daily Revenue vs. 7-Day Moving Average"
)
ax.set_xticks(ax.get_xticks()[::5])


### 2.6 View for a frequently used query

In [ ]:
execute("""
CREATE VIEW IF NOT EXISTS vw_monthly_category_revenue AS
SELECT
    order_month,
    category,
    COUNT(*) AS order_count,
    SUM(amount) AS total_revenue
FROM sales
WHERE amount > 0
GROUP BY order_month, category
""")

run_query("SELECT * FROM vw_monthly_category_revenue ORDER BY order_month, total_revenue DESC LIMIT 15")


### 2.7 Query optimization: EXPLAIN QUERY PLAN + indexing
`scripts/load_data.py` already creates `idx_sales_status`, `idx_sales_category`, `idx_sales_date`, `idx_sales_state`, and `idx_sales_order_month`. The plan below should show SQLite using `idx_sales_status` (`SEARCH sales USING INDEX ...`) rather than scanning all 128,942 rows.

In [ ]:
explain("SELECT * FROM sales WHERE status = 'Cancelled'")


## 3. Python + SQL Integration (Day 15-17)

`scripts/db_utils.py` is the reusable connection module every query above already goes through:

- `get_engine()` — creates (and caches) a SQLAlchemy engine. Points at SQLite by default; set the `DATABASE_URL`
  env var to point the *same* functions at Postgres or MySQL instead.
- `run_query(sql, params)` — runs a parameterized `SELECT` and returns a DataFrame via `pandas.read_sql`.
- `execute(sql, params)` — runs a non-`SELECT` statement (`CREATE`, `INSERT`, ...).
- `explain(sql)` — wraps a query in `EXPLAIN QUERY PLAN`.

All the query calls above already used **named parameters** (`:category`, `:n`, etc.) instead of
f-string/`.format()` interpolation — that's what keeps them safe from SQL injection even when a
value comes from user input.

### Why parameterized queries matter
Compare a naive f-string query to the parameterized version:

In [ ]:
# UNSAFE (don't do this) -- if `user_input` ever contains a value like
# "Set' OR '1'='1", the WHERE clause is silently rewritten.
user_input = "Set"
unsafe_sql = f"SELECT COUNT(*) AS n FROM sales WHERE category = '{user_input}'"
print(unsafe_sql)

# SAFE -- the driver escapes the value; it can never change the query's structure.
safe_result = run_query("SELECT COUNT(*) AS n FROM sales WHERE category = :cat", {"cat": user_input})
safe_result


### End-to-end example: parameterized function returning a DataFrame

In [ ]:
def category_orders_in_month(category: str, month: str) -> pd.DataFrame:
    """Orders for one category in one order_month, via a parameterized query."""
    sql = """
        SELECT order_id, date, qty, amount, ship_state
        FROM sales
        WHERE category = :category AND order_month = :month
        ORDER BY amount DESC
    """
    return run_query(sql, {"category": category, "month": month})

category_orders_in_month("Saree", "2022-05").head(10)


## Summary

- Ran the full range of required SQL (`SELECT`/`WHERE`/`ORDER BY`/`LIMIT`, all four JOIN types, `GROUP BY`/`HAVING`,
  subqueries, CTEs, and window functions) against the cleaned Amazon Sale Report data.
- Answered the five business questions from the brief: monthly sales trend, top revenue-generating ship-to
  locations (customer-proxy), a location-level retention rate, category performance, and 7-day moving
  average / cumulative revenue.
- Built and used a reusable `db_utils.py` module and a saved view (`vw_monthly_category_revenue`), and confirmed
  indexes are actually used via `EXPLAIN QUERY PLAN`.
- Every query above runs through parameterized SQL rather than string interpolation.

**Caveat carried into Task 3/4:** "customer" metrics in this notebook are a `ship_city` + `ship_postal_code`
proxy, since the Amazon Sale Report export has no `customer_id`. Worth flagging in the final report.